# MMLU Solver/Judge Difficulty Comparison

Load MMLU K-factor item difficulty JSONs and keep each item's per-model scores. `mmlu_solver` is solver-side; `mmlu_judging` is judge-side. MMLU judging has `original` and `swapped` rows, so judging difficulty and judging scores are averaged to one row per `pair_id` before joining to solver items.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 240)


## Choose K And Resolve Paths


In [ ]:
K = 2  # choose 1 or 2


def find_repo_root(start=Path.cwd()):
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "K-Factor").exists() and (candidate / "benchmarks").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing K-Factor/ and benchmarks/")


REPO_ROOT = find_repo_root()
print(f"Repo root: {REPO_ROOT}")

solver_json_path = REPO_ROOT / "K-Factor" / "results" / "mmlu_solver" / "mmlu_solver_kfactor_item_difficulties_with_laplace_uncertainty.json"
judge_json_path = REPO_ROOT / "K-Factor" / "results" / "mmlu_judging" / "mmlu_judging_kfactor_item_difficulties_with_laplace_uncertainty.json"
solver_fit_summary_path = REPO_ROOT / "K-Factor" / "results" / "mmlu_solver" / "mmlu_solver_kfactor_fit_summary.csv"
judge_fit_summary_path = REPO_ROOT / "K-Factor" / "results" / "mmlu_judging" / "mmlu_judging_kfactor_fit_summary.csv"
question_path = REPO_ROOT / "benchmarks" / "mmlu" / "judgebench_mmlu_pro_questions_unique.csv"

out_dir = REPO_ROOT / "K-Factor" / "results" / "mmlu_solver_judge_comparison"
out_dir.mkdir(parents=True, exist_ok=True)

for path in [solver_json_path, judge_json_path, solver_fit_summary_path, judge_fit_summary_path, question_path]:
    print(path.relative_to(REPO_ROOT), path.exists())


## Fit Summaries


In [ ]:
solver_fit_summary = pd.read_csv(solver_fit_summary_path)
judge_fit_summary = pd.read_csv(judge_fit_summary_path)

def choose_best_k(summary):
    return (
        summary
        .sort_values(["auc", "ece", "loss"], ascending=[False, True, True])
        .iloc[0]
    )

solver_best = choose_best_k(solver_fit_summary)
judge_best = choose_best_k(judge_fit_summary)

print(f"Solver-side best k by AUC (tie-break ECE/loss): k={int(solver_best['k'])}")
print(f"Judge-side best k by AUC (tie-break ECE/loss):  k={int(judge_best['k'])}")
print(f"Currently loading K={K}")

print()
print("Solver fit summary")
display(solver_fit_summary.round(4))

print("Judge fit summary")
display(judge_fit_summary.round(4))

solver_fit_summary.to_csv(out_dir / "mmlu_solver_kfactor_fit_summary.csv", index=False)
judge_fit_summary.to_csv(out_dir / "mmlu_judge_kfactor_fit_summary.csv", index=False)


## Load Item Difficulties And Scores


In [ ]:
def load_kfactor_fit(json_path, k, score_col):
    with open(json_path) as f:
        payload = json.load(f)

    fit_key = f"k{k}"
    if fit_key not in payload["fits"]:
        raise KeyError(f"{json_path} does not contain {fit_key}; available={list(payload['fits'])}")

    df = pd.DataFrame(payload["fits"][fit_key])
    df["item_id"] = df["item_id"].astype(str)
    df["pair_id"] = df["pair_id"].astype(str)
    df = df.rename(columns={"scores": score_col})
    return df, payload


solver_items, solver_payload = load_kfactor_fit(solver_json_path, K, "solver_scores")
judge_order_items, judge_payload = load_kfactor_fit(judge_json_path, K, "judge_order_scores")

print(f"solver_items:      {solver_items.shape}")
print(f"judge_order_items: {judge_order_items.shape}")

solver_items.to_csv(out_dir / f"mmlu_k{K}_solver_items.csv", index=False)
solver_items.to_json(out_dir / f"mmlu_k{K}_solver_items.json", orient="records", indent=2, force_ascii=False)
judge_order_items.to_csv(out_dir / f"mmlu_k{K}_judge_order_items.csv", index=False)
judge_order_items.to_json(out_dir / f"mmlu_k{K}_judge_order_items.json", orient="records", indent=2, force_ascii=False)


## Attach Prompt Text


In [ ]:
def split_pair_ids(value):
    if pd.isna(value) or str(value).strip() == "":
        return []
    text = str(value)
    for sep in [";", "|"]:
        text = text.replace(sep, ",")
    return [x.strip() for x in text.split(",") if x.strip()]

questions = pd.read_csv(question_path)
prompt_rows = []
for _, row in questions.iterrows():
    for pair_col, label_col, split_name in [
        ("gpt_pair_ids", "gpt_labels", "gpt"),
        ("claude_pair_ids", "claude_labels", "claude"),
    ]:
        for pair_id in split_pair_ids(row.get(pair_col)):
            prompt_rows.append({
                "pair_id": pair_id,
                "question_index": row.get("question_index"),
                "original_ids": row.get("original_ids"),
                "question_source": row.get("source"),
                "question_split": split_name,
                "gold_label": row.get(label_col),
                "prompt": row.get("question"),
            })

prompt_lookup = pd.DataFrame(prompt_rows).drop_duplicates("pair_id")
print(f"prompt_lookup: {prompt_lookup.shape}")

solver_items = solver_items.merge(prompt_lookup, on="pair_id", how="left")
judge_order_items = judge_order_items.merge(
    prompt_lookup.drop(columns=["gold_label"], errors="ignore"),
    on="pair_id",
    how="left",
    suffixes=("", "_prompt"),
)

print(f"solver prompts missing: {solver_items['prompt'].isna().sum()}")
print(f"judge prompts missing:  {judge_order_items['prompt'].isna().sum()}")

display(solver_items.head())


## Aggregate Judging Original/Swapped Rows


In [ ]:
def mean_score_dict(dicts):
    models = sorted({model for scores in dicts if isinstance(scores, dict) for model in scores})
    averaged = {}
    for model in models:
        vals = []
        for scores in dicts:
            if isinstance(scores, dict):
                val = scores.get(model)
                if val is not None and not pd.isna(val):
                    vals.append(float(val))
        averaged[model] = float(np.mean(vals)) if vals else None
    return averaged


def combine_se_for_mean(values):
    vals = pd.Series(values).dropna().astype(float)
    if vals.empty:
        return np.nan
    return float(np.sqrt(np.square(vals).sum()) / len(vals))


def aggregate_judge_pair(group):
    return pd.Series({
        "judge_item_ids": list(group["item_id"].astype(str)),
        "judge_orders": list(group["order"].astype(str)),
        "judge_source": group["source"].iloc[0],
        "judge_split": group["split"].iloc[0],
        "judge_response_model": group["response_model"].iloc[0],
        "judge_gold_labels": dict(zip(group["order"].astype(str), group["gold_label"].astype(str))),
        "judge_difficulty": group["difficulty"].mean(),
        "judge_difficulty_centered": group["difficulty_centered"].mean(),
        "judge_difficulty_laplace_se": combine_se_for_mean(group["difficulty_laplace_se"]),
        "judge_difficulty_centered_laplace_se": combine_se_for_mean(group["difficulty_centered_laplace_se"]),
        "judge_dominant_factors": dict(zip(group["order"].astype(str), group["dominant_factor"].astype(str))),
        "judge_scores": mean_score_dict(group["judge_order_scores"]),
        "prompt": group["prompt"].dropna().iloc[0] if group["prompt"].notna().any() else None,
    })


judge_items = (
    judge_order_items
    .groupby("pair_id", as_index=False)
    .apply(aggregate_judge_pair, include_groups=False)
    .reset_index(drop=True)
)

print(f"judge_items aggregated: {judge_items.shape}")
display(judge_items.head())

judge_items.to_csv(out_dir / f"mmlu_k{K}_judge_items_aggregated.csv", index=False)
judge_items.to_json(out_dir / f"mmlu_k{K}_judge_items_aggregated.json", orient="records", indent=2, force_ascii=False)


## Join Solver And Judge Items


In [ ]:
def prefix_solver_cols(df):
    return df.rename(columns={col: f"solver_{col}" for col in df.columns if col != "solver_scores"})


def prefix_judge_cols(df):
    rename = {}
    for col in df.columns:
        if col == "judge_scores":
            continue
        if col == "pair_id":
            rename[col] = "judge_pair_id"
        elif col.startswith("judge_"):
            continue
        else:
            rename[col] = f"judge_{col}"
    return df.rename(columns=rename)


solver_for_join = prefix_solver_cols(solver_items)
judge_for_join = prefix_judge_cols(judge_items)

paired_items = solver_for_join.merge(
    judge_for_join,
    left_on="solver_pair_id",
    right_on="judge_pair_id",
    how="inner",
    validate="one_to_one",
)

print(f"paired_items: {paired_items.shape}")
display(paired_items.head())

paired_items.to_csv(out_dir / f"mmlu_k{K}_paired_items.csv", index=False)
paired_items.to_json(out_dir / f"mmlu_k{K}_paired_items.json", orient="records", indent=2, force_ascii=False)


## Part One: Solver Difficulty vs Mean Judging Score


In [ ]:
partone_table = paired_items[[
    "solver_item_id",
    "solver_pair_id",
    "solver_source",
    "solver_difficulty_centered",
    "solver_difficulty_centered_laplace_se",
    "solver_prompt",
    "solver_gold_letter",
    "solver_scores",
    "judge_scores",
]].copy()

partone_table["judge_score_mean"] = partone_table["judge_scores"].apply(
    lambda scores: pd.Series(scores).dropna().mean()
)

partone_table_sorted = partone_table.sort_values("solver_difficulty_centered", ascending=False)

display(partone_table_sorted.head())

partone_table.to_csv(out_dir / f"mmlu_k{K}_partone_table_with_judge_score_mean.csv", index=False)
partone_table.to_json(out_dir / f"mmlu_k{K}_partone_table_with_judge_score_mean.json", orient="records", indent=2, force_ascii=False)
partone_table_sorted.to_csv(out_dir / f"mmlu_k{K}_partone_table_sorted.csv", index=False)
partone_table_sorted.to_json(out_dir / f"mmlu_k{K}_partone_table_sorted.json", orient="records", indent=2, force_ascii=False)


## Judge Score By Solver Difficulty Decile


In [ ]:
plot_df = partone_table.copy()

plot_df["difficulty_bin"] = pd.qcut(
    plot_df["solver_difficulty_centered"],
    q=10,
    labels=False,
    duplicates="drop",
)

bin_summary = (
    plot_df
    .groupby("difficulty_bin", observed=True)
    .agg(
        solver_difficulty_mean=("solver_difficulty_centered", "mean"),
        judge_score_mean=("judge_score_mean", "mean"),
        n_items=("solver_item_id", "count"),
    )
    .reset_index()
)

bin_summary["difficulty_bin"] = bin_summary["difficulty_bin"] + 1

plt.figure(figsize=(8, 5))
plt.bar(bin_summary["difficulty_bin"], bin_summary["judge_score_mean"])
plt.xlabel("Solver difficulty quantile bin")
plt.ylabel("Mean judge score")
plt.title("Mean judge score across solver difficulty deciles")
plt.xticks(bin_summary["difficulty_bin"])
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(out_dir / f"mmlu_k{K}_difficulty_bin_summary.png", dpi=300, bbox_inches="tight")
plt.savefig(out_dir / f"mmlu_k{K}_difficulty_bin_summary.pdf", bbox_inches="tight")
plt.show()

display(bin_summary)

bin_summary.to_csv(out_dir / f"mmlu_k{K}_difficulty_bin_summary.csv", index=False)
bin_summary.to_json(out_dir / f"mmlu_k{K}_difficulty_bin_summary.json", orient="records", indent=2, force_ascii=False)


In [ ]:
# Per-judge-model view of Part 1.
# Each box shows the distribution across judge models within a solver difficulty bin.
model_plot_df = partone_table.copy()
model_plot_df["difficulty_bin"] = pd.qcut(
    model_plot_df["solver_difficulty_centered"],
    q=10,
    labels=False,
    duplicates="drop",
)
model_plot_df["difficulty_bin"] = model_plot_df["difficulty_bin"] + 1

judge_model_rows = []
for _, row in model_plot_df.iterrows():
    scores = row["judge_scores"]
    if not isinstance(scores, dict):
        continue
    for judge_model, score in scores.items():
        if score is None or pd.isna(score):
            continue
        judge_model_rows.append({
            "difficulty_bin": int(row["difficulty_bin"]),
            "solver_item_id": row["solver_item_id"],
            "judge_model": judge_model,
            "judge_score": float(score),
        })

judge_model_scores = pd.DataFrame(judge_model_rows)
judge_model_bin_summary = (
    judge_model_scores
    .groupby(["difficulty_bin", "judge_model"], observed=True)
    .agg(
        judge_score_mean=("judge_score", "mean"),
        n_items=("solver_item_id", "count"),
    )
    .reset_index()
)

judge_model_bin_mean = (
    judge_model_bin_summary
    .groupby("difficulty_bin", observed=True)
    .agg(
        judge_score_mean=("judge_score_mean", "mean"),
        judge_score_sd=("judge_score_mean", "std"),
        n_judge_models=("judge_model", "count"),
    )
    .reset_index()
)

bins = sorted(judge_model_bin_summary["difficulty_bin"].unique())
box_values = [
    judge_model_bin_summary.loc[
        judge_model_bin_summary["difficulty_bin"] == bin_id,
        "judge_score_mean",
    ].dropna().to_numpy()
    for bin_id in bins
]

rng = np.random.default_rng(123)
plot_x = judge_model_bin_summary["difficulty_bin"].to_numpy(dtype=float)
plot_x = plot_x + rng.uniform(-0.14, 0.14, size=len(plot_x))

plt.figure(figsize=(9, 5.8))
plt.boxplot(
    box_values,
    positions=bins,
    widths=0.55,
    patch_artist=True,
    showfliers=False,
    boxprops={"facecolor": "0.88", "edgecolor": "0.35", "linewidth": 1.0},
    medianprops={"color": "0.2", "linewidth": 1.4},
    whiskerprops={"color": "0.45", "linewidth": 1.0},
    capprops={"color": "0.45", "linewidth": 1.0},
)
plt.scatter(
    plot_x,
    judge_model_bin_summary["judge_score_mean"],
    s=20,
    color="0.35",
    alpha=0.55,
    label="Judge model",
)
plt.plot(
    judge_model_bin_mean["difficulty_bin"],
    judge_model_bin_mean["judge_score_mean"],
    color="black",
    marker="D",
    markersize=5,
    linewidth=1.8,
    label="Mean across judge models",
)
plt.xlabel("Solver difficulty quantile bin")
plt.ylabel("Mean judge score")
plt.title("Judge model score distribution across solver difficulty deciles")
plt.xticks(bins)
plt.grid(axis="y", alpha=0.3)
plt.legend(loc="best")
plt.figtext(
    0.5,
    0.01,
    "Each box shows the distribution across judge models within a solver difficulty bin.",
    ha="center",
    fontsize=9,
)
plt.tight_layout(rect=(0, 0.04, 1, 1))
plt.savefig(out_dir / f"mmlu_k{K}_judge_model_bin_summary.png", dpi=300, bbox_inches="tight")
plt.savefig(out_dir / f"mmlu_k{K}_judge_model_bin_summary.pdf", bbox_inches="tight")
plt.show()

display(judge_model_bin_summary)
display(judge_model_bin_mean)

judge_model_bin_summary.to_csv(out_dir / f"mmlu_k{K}_judge_model_bin_summary.csv", index=False)
judge_model_bin_summary.to_json(out_dir / f"mmlu_k{K}_judge_model_bin_summary.json", orient="records", indent=2, force_ascii=False)
judge_model_bin_mean.to_csv(out_dir / f"mmlu_k{K}_judge_model_bin_summary_mean.csv", index=False)
judge_model_bin_mean.to_json(out_dir / f"mmlu_k{K}_judge_model_bin_summary_mean.json", orient="records", indent=2, force_ascii=False)


## Part Two: Solver Difficulty vs Judge Difficulty


In [ ]:
parttwo_table = paired_items[[
    "solver_item_id",
    "solver_pair_id",
    "solver_source",
    "solver_difficulty_centered",
    "solver_difficulty_centered_laplace_se",
    "solver_gold_letter",
    "solver_scores",
    "solver_prompt",
    "judge_difficulty_centered",
    "judge_difficulty_centered_laplace_se",
    "judge_scores",
]].copy()

parttwo_table_sorted = parttwo_table.sort_values("solver_difficulty_centered", ascending=False)

display(parttwo_table_sorted.head())

parttwo_table.to_csv(out_dir / f"mmlu_k{K}_parttwo_table_raw.csv", index=False)
parttwo_table.to_json(out_dir / f"mmlu_k{K}_parttwo_table_raw.json", orient="records", indent=2, force_ascii=False)
parttwo_table_sorted.to_csv(out_dir / f"mmlu_k{K}_parttwo_table_sorted.csv", index=False)
parttwo_table_sorted.to_json(out_dir / f"mmlu_k{K}_parttwo_table_sorted.json", orient="records", indent=2, force_ascii=False)


## Solver Difficulty vs Judge Difficulty Plot


In [ ]:
parttwo_difficulty_df = parttwo_table[[
    "solver_item_id",
    "solver_pair_id",
    "solver_source",
    "solver_difficulty_centered",
    "solver_difficulty_centered_laplace_se",
    "judge_difficulty_centered",
    "judge_difficulty_centered_laplace_se",
]].dropna().copy()

raw_rho, raw_p = spearmanr(
    parttwo_difficulty_df["solver_difficulty_centered"],
    parttwo_difficulty_df["judge_difficulty_centered"],
)
print(f"Spearman rho: {raw_rho:.4f}")
print(f"p-value:      {raw_p:.4g}")

plt.figure(figsize=(6, 6))
plt.errorbar(
    parttwo_difficulty_df["solver_difficulty_centered"],
    parttwo_difficulty_df["judge_difficulty_centered"],
    xerr=parttwo_difficulty_df["solver_difficulty_centered_laplace_se"],
    yerr=parttwo_difficulty_df["judge_difficulty_centered_laplace_se"],
    fmt="o",
    markersize=3,
    alpha=0.55,
    ecolor="gray",
    elinewidth=0.6,
    capsize=1.5,
)

plt.axhline(0, color="black", linewidth=1, alpha=0.35)
plt.axvline(0, color="black", linewidth=1, alpha=0.35)
plt.xlabel("Solver difficulty centered")
plt.ylabel("Judge difficulty centered")
plt.title(f"Solver vs Judge Difficulty (rho={raw_rho:.3f}, p={raw_p:.3g})")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(out_dir / f"mmlu_k{K}_solver_judge_difficulty_scatter.png", dpi=300, bbox_inches="tight")
plt.savefig(out_dir / f"mmlu_k{K}_solver_judge_difficulty_scatter.pdf", bbox_inches="tight")
plt.show()

display(parttwo_difficulty_df.head())

parttwo_difficulty_df.to_csv(out_dir / f"mmlu_k{K}_solver_judge_difficulty_scatter.csv", index=False)
parttwo_difficulty_df.to_json(out_dir / f"mmlu_k{K}_solver_judge_difficulty_scatter.json", orient="records", indent=2, force_ascii=False)


## Solver Percentile vs Judge Percentile


In [ ]:
percentile_df = parttwo_difficulty_df.copy()
percentile_df["solver_difficulty_percentile"] = percentile_df["solver_difficulty_centered"].rank(pct=True) * 100
percentile_df["judge_difficulty_percentile"] = percentile_df["judge_difficulty_centered"].rank(pct=True) * 100

pct_rho, pct_p = spearmanr(
    percentile_df["solver_difficulty_percentile"],
    percentile_df["judge_difficulty_percentile"],
)
print(f"Percentile Spearman rho: {pct_rho:.4f}")
print(f"Percentile p-value:      {pct_p:.4g}")

plt.figure(figsize=(6, 6))
plt.scatter(
    percentile_df["solver_difficulty_percentile"],
    percentile_df["judge_difficulty_percentile"],
    s=14,
    alpha=0.65,
)
plt.plot([0, 100], [0, 100], color="black", linewidth=1, alpha=0.4)
plt.xlabel("Solver difficulty percentile")
plt.ylabel("Judge difficulty percentile")
plt.title(f"Solver vs Judge Difficulty Percentiles (rho={pct_rho:.3f}, p={pct_p:.3g})")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.savefig(out_dir / f"mmlu_k{K}_solver_judge_difficulty_percentiles.png", dpi=300, bbox_inches="tight")
plt.savefig(out_dir / f"mmlu_k{K}_solver_judge_difficulty_percentiles.pdf", bbox_inches="tight")
plt.show()

display(percentile_df.head())

percentile_df.to_csv(out_dir / f"mmlu_k{K}_solver_judge_difficulty_percentiles.csv", index=False)
percentile_df.to_json(out_dir / f"mmlu_k{K}_solver_judge_difficulty_percentiles.json", orient="records", indent=2, force_ascii=False)
